# Anomalie di temperatura nelle Isole Britanniche (Regno Unito + Irlanda) e confronto con l'AMO Index

Questo notebook ricostruisce la serie storica delle **anomalie di temperatura**
delle Isole Britanniche a partire dai dati di stazione **CRUTEM 4.6.0.0** e la
confronta con l'**AMO Index** (Atlantic Multidecadal Oscillation) calcolato dai
dati osservati di temperatura superficiale del mare **HadISST**.

Rispetto alla versione iniziale (solo Regno Unito, indice AMO da pressione) sono
state introdotte tre modifiche metodologiche, documentate cella per cella:

1. **inclusione delle stazioni irlandesi** per aumentare il campione;
2. **serie regionale unica** al posto della griglia 10°, che produceva celle con
   pochissime stazioni;
3. **AMO Index calcolato dalla SST (HadISST)**, cioè secondo la definizione
   fisica corretta dell'oscillazione, al posto della pressione al livello del mare.

Ogni cella di codice è preceduta da una spiegazione (cosa fa e perché) e, dove
produce un risultato, seguita da un commento sui risultati.

## Librerie

Si importano le librerie scientifiche usate nel notebook: `numpy`/`pandas` per il
calcolo numerico e le serie temporali, `geopandas`/`cartopy` per la cartografia,
`xarray` per i dati grigliá NetCDF (HadISST), `matplotlib` per i grafici e
`scipy` per le statistiche.

In [ ]:
# Libraries

import os
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd

from datetime import datetime
from tqdm import tqdm
import datetime as dtm

import matplotlib.pyplot as plt
import cartopy.feature as cfeature
import cartopy.crs as ccrs
import cartopy.feature as cf

import xarray as xr

from scipy import stats

## Lettura del dataset CRUTEM

**Cosa fa:** elenca tutti i file delle stazioni meteorologiche contenuti
nell'archivio CRUTEM e conta quante stazioni sono disponibili.
**Perché:** ogni file ASCII contiene i metadati e la serie mensile di una
stazione; serve la lista completa per poterli leggere uno a uno.

In [ ]:
# per capire come è costituito il dataset
flist = [os.path.join(path, name) 
         for path, subdirs, 
         files in os.walk("C:\\Users\\kekko\\Downloads\\Python\\Lab of Geo\\CRUTEM.4.6.0.0.station_files") 
         for name in files]  # contiene i dati grezzi derivanti da una serie di stazioni meteorologiche

for i in range(0,5):
  print (flist[i])

# exclude first 2 items, not relevant
flist=flist[2:]

print ('\n',flist[0:5])
nst=len(flist)
print ("\n > Number of stations = ",nst)  # si hanno più di 10k stazioni

*Risultato:* l'archivio contiene oltre **10.000 stazioni** in tutto il mondo: da
qui verranno poi selezionate solo quelle delle Isole Britanniche.

## Aggiunta delle stazioni irlandesi

**Cosa fa / perché:** per **aumentare il numero di stazioni** e migliorare la
copertura spaziale, l'analisi viene estesa dalle sole stazioni del **Regno Unito**
anche a quelle dell'**Irlanda**. Le Isole Britanniche sono un'area climaticamente
omogenea (clima oceanico temperato), quindi trattarle insieme è coerente dal
punto di vista geografico e statistico.

La cella diagnostica seguente campiona un sottoinsieme di file e individua le
**stringhe esatte** del campo paese, sia per il Regno Unito sia per l'Irlanda, da
usare nel filtro della cella di lettura dati.

In [ ]:
# --- VERIFICA STRINGHE PAESE (eseguire una sola volta, prima del loop completo) ---
# Il campo 'Country' nei file CRUTEM non e' standardizzato in modo univoco:
# a seconda della versione del dataset puo' comparire come 'UK', 'UNITED KINGDOM',
# 'IRELAND', 'IRISH REPUBLIC', ecc. Oltre al Regno Unito vogliamo ora includere
# anche l'Irlanda per aumentare il numero di stazioni disponibili. Qui si campiona
# un sottoinsieme di file per individuare le stringhe esatte (UK e Irlanda) da
# usare nel filtro della cella successiva.

found_countries = set()
for filein in flist[:3000]:  # campione, non l'intero dataset
    with open(filein, 'r') as f:
        for line in f:
            if line.startswith('Country'):
                found_countries.add(line.split('=', 1)[1].strip())
                break

uk_like = sorted(c for c in found_countries
                 if 'UK' in c.upper() or 'KINGDOM' in c.upper() or 'BRITAIN' in c.upper())
ie_like = sorted(c for c in found_countries
                 if 'IRELAND' in c.upper() or 'EIRE' in c.upper() or 'IRISH' in c.upper())
print('Possibili varianti trovate per UK:     ', uk_like)
print('Possibili varianti trovate per Irlanda:', ie_like)
print('\n> Copia le stringhe esatte nella variabile TARGET_COUNTRIES nella cella successiva.')

*Risultato:* la diagnostica conferma le stringhe presenti nel dataset (`UK` e
`IRELAND`), che vengono usate nell'insieme `TARGET_COUNTRIES` del filtro.

## Strutture dati per metadati e serie

**Cosa fa:** crea il DataFrame vuoto dei **metadati** (ID, nome, paese, quota,
lat, lon). Gli ID stazione fungono da identificatore univoco.

In [ ]:
# The first challenge is to correctly read in the data from an individual ASCII files.

# Define df to store metadata.
# Station IDs will be used as unique identifiers for both dataframes.

metadata0 = pd.DataFrame(columns=[
    'ID',
    'stname',
    'country',
    'elev',
    'lat',
    'lon'])

## Asse temporale comune e periodo di riferimento

**Cosa fa:** definisce un asse temporale mensile comune (1850–2018) su cui
allineare tutte le stazioni e fissa il **periodo di riferimento** per il calcolo
delle normali climatiche.
**Perché:** stazioni diverse coprono intervalli diversi; un asse comune permette
di combinarle e confrontarle correttamente.

In [ ]:
# define a common time axis
taxis = pd.date_range('1850-01', '2019-01', freq='ME')
nmonths=len(taxis)
nyears=nmonths/12

# Define df to store data (temperature time series) and location. 
data0 = pd.DataFrame({'time': taxis})
data0 = data0.set_index(['time'])

# Define reference period
yref0 = 1951
yref1 = 2010

## Lettura e filtraggio delle stazioni

**Cosa fa:** scorre tutti i file, ne estrae metadati e serie, e applica i filtri:
paese in `{'UK','IRELAND'}`, normali valide e almeno 30 anni di dati.
**Perché:** si selezionano solo stazioni delle Isole Britanniche con serie
sufficientemente lunghe per essere rappresentative del clima locale.

In [ ]:
# Liste per accumulare i dati (molto più veloce di append/merge in loop)
metadata_list = []
data_series_list = []

nodatacount = tooshortcount = outsidecount = 0

for filein in tqdm(flist[:nst]):
    with open(filein, 'r') as f:
        lines = f.readlines()

    # Estrazione rapida metadati in un colpo solo
    header = {}
    obs_index = -1
    for i, line in enumerate(lines):
        if '=' in line:
            k, v = line.split('=', 1)
            header[k.strip()] = v.strip()
        if "Obs:" in line:
            obs_index = i + 1
            break

    # 1. Filtro esistenza dati
    if obs_index == -1 or len(lines) <= obs_index:
        nodatacount += 1
        continue

    # 2. Filtro Country e Normals (rapidi)
    # NB: usare le stringhe esatte individuate con la cella diagnostica precedente.
    # Includiamo sia il Regno Unito sia l'Irlanda per aumentare il numero di stazioni.
    TARGET_COUNTRIES = {'UK', 'IRELAND'}  # <-- aggiornare in base all'output della cella diagnostica
    if header.get('Country', '').strip().upper() not in TARGET_COUNTRIES:
        continue
    
    if "-99.0" in header.get('Normals', ''):
        outsidecount += 1
        continue

    # Lettura dati numerici (solo una volta)
    try:
        raw_data = np.loadtxt(lines[obs_index:], dtype='f4')
        if raw_data.ndim == 1: raw_data = raw_data.reshape(1, -1) # Fix per file con 1 sola riga
    except:
        continue

    # 3. Filtro durata (anni)
    nyr = len(raw_data)
    if nyr < 30:
        tooshortcount += 1
        continue

    # Salvataggio Metadati
    st_id = filein[-6:]
    metadata_list.append({
        "ID": st_id, "nome": header.get('Name'), "paese": header.get('Country'),
        "altezza": float(header.get('Height', 0)), "lat": float(header.get('Lat', 0)), 
        "lon": float(header.get('Long', 0))
    })

    # Preparazione serie temporale
    # flatten() dei dati escludendo la prima colonna (l'anno)
    values = raw_data[:, 1:13].flatten()
    
    ti0 = datetime(int(raw_data[0, 0]), 1, 1)
    stime = pd.date_range(ti0, periods=len(values), freq='ME') # 'ME' è il nuovo standard per Month End
    
    data_series_list.append(pd.Series(values, index=stime, name=st_id))

# Creazione DataFrame finali (una sola operazione distruttiva alla fine)
metadata0 = pd.DataFrame(metadata_list)
data0 = pd.concat(data_series_list, axis=1)

*Risultato:* al termine del loop si ottengono i DataFrame `metadata0` (metadati) e
`data0` (serie mensili allineate).

## Riepilogo del filtraggio

**Cosa fa:** stampa quante stazioni sono state scartate (assenza dati, serie
troppo corte, normali non valide) e quante sono state mantenute.

In [ ]:
# --- Summary ---
print(f"Skipped - no data:     {nodatacount}")
print(f"Skipped - too short:   {tooshortcount}")
print(f"Skipped - outside ref: {outsidecount}")
print(f"Stations retained:     {len(metadata0)}")

*Risultato:* l'inclusione dell'Irlanda porta a **64 stazioni** mantenute (contro
le 44 della sola versione UK): il campione regionale è ora più ricco.

## Metadati delle stazioni selezionate

**Cosa fa:** mostra la tabella dei metadati delle stazioni mantenute.

In [ ]:
# Display metadata and data
print("\nMetadata:")
print(metadata0)

The CRUTEM dataset contains thousands of global stations, and applying filters
(country = UK or Ireland, at least 30 years of data, valid normals) yields a
subset of stations over the British Isles. This ensures that the analyzed time
series are long enough to be statistically representative of the local climate.

## Correzione della longitudine

**Cosa fa:** inverte il segno della longitudine.
**Perché:** nei file CRUTEM la longitudine è memorizzata come positiva verso
ovest; per la cartografia standard (est positivo) va invertita.

In [ ]:
# correzione della longitudine
metadata0.lon = -metadata0.lon

## Posizione delle stazioni

**Cosa fa:** disegna le stazioni su una vista globale e su uno zoom delle Isole
Britanniche (esteso a ovest fino a −11° per includere l'Irlanda).

In [ ]:
# Visualizzazione stazione
lon, lat = metadata0.lon, metadata0.lat
pc = ccrs.PlateCarree()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4),
                                subplot_kw=dict(projection=pc))

# --- Vista globale ---
ax1.set_title("Vista globale")
ax1.stock_img()
ax1.coastlines()

# --- Zoom UK ---
ax2.set_title("Zoom Regno Unito e Irlanda")
ax2.set_extent([-11, 2, 49.5, 61], crs=pc)  # bounding box Isole Britanniche (UK + Irlanda)
ax2.coastlines(resolution='10m')
for feat, kw in [
    (cfeature.BORDERS, dict(linestyle=':')),
    (cfeature.LAND,   dict(facecolor='lightgray')),
    (cfeature.OCEAN,  dict(facecolor='lightblue')),
]:
    ax2.add_feature(feat, **kw)
ax2.gridlines(draw_labels=True, linestyle='--', alpha=0.5)

# scatter su entrambi gli assi
for ax, s in [(ax1, 10), (ax2, 50)]:
    ax.scatter([lon], [lat], color='r', s=s, transform=pc)

plt.tight_layout()
plt.show()

*Risultato:* le stazioni coprono Gran Bretagna e Irlanda in modo abbastanza
uniforme, confermando la buona copertura spaziale dopo l'aggiunta irlandese.

## Pulizia dei valori mancanti

**Cosa fa:** sostituisce il valore convenzionale `-99.0` con `NaN`.
**Perché:** i dati mancanti non devono essere trattati come temperature reali nei
calcoli successivi (medie, anomalie).

In [ ]:
# Sostituisce i -99 con NaN in tutto il DataFrame
# Tutte le colonne vengono mantenute, nessuna viene eliminata.
data0 = data0.replace(-99.0, np.nan)


## Serie grezze per stazione

**Cosa fa:** per ogni stazione affianca la sua posizione su mappa e la sua serie
temporale mensile grezza.
**Perché:** controllo visivo della qualità e copertura temporale delle singole
serie prima dell'analisi delle anomalie.

In [ ]:
# Grafico combinato: mappa + serie temporale per ogni stazione
n = len(metadata0)
fig = plt.figure(figsize=(8, 2.5 * n))  

for i, (_, row) in enumerate(metadata0.iterrows()):
    # Mappa (sinistra)
    ax_map = plt.subplot(n, 2, i * 2 + 1, projection=ccrs.PlateCarree())
    ax_map.set_title(f"{row.nome} - {row.paese}", fontsize=9)
    ax_map.add_feature(cf.LAND, facecolor="lightgray")
    ax_map.add_feature(cf.OCEAN, facecolor="lightblue")
    ax_map.coastlines(linewidth=0.2)
    ax_map.plot(row.lon, row.lat, "ro", markersize=5, transform=ccrs.Geodetic())

    # Serie temporale (destra)
    ax_ts = plt.subplot(n, 2, i * 2 + 2)
    st_id = row["ID"]
    if st_id in data0.columns:
        ax_ts.plot(data0[st_id], color='tab:blue', lw=0.5)
        ax_ts.set_title(f"ID Stazione: {st_id}")
        ax_ts.grid(True, alpha=0.3)
    else:
        ax_ts.text(0.5, 0.5, f"Dati per {st_id} non trovati", ha='center', transform=ax_ts.transAxes)

plt.tight_layout(pad=1.2)
plt.show()

*Risultato:* le serie mostrano lunghezze e completezze diverse; molte coprono gran
parte del periodo 1850–2018, alcune presentano lacune.

## Indicizzazione per ID

**Cosa fa:** imposta l'ID stazione come indice del DataFrame dei metadati, per
collegarlo facilmente alle colonne di `data0`.

In [ ]:
metadata0 = metadata0.set_index(["ID"])

## Calcolo delle normali climatiche

**Cosa fa:** calcola, per ciascun mese, la temperatura media nel periodo di
riferimento (1951–2000).
**Perché:** le normali rappresentano il ciclo stagionale medio, che verrà
sottratto per ottenere le anomalie.

In [ ]:
x1=data0.index.get_loc('1951-01-31')
x2=data0.index.get_loc('2000-12-31')

# calcolo delle normals, si è fatta la media di tutti i gennaio dal 1950 al 2000 
data_normals = data0[x1:x2].groupby(data0[x1:x2].index.month).mean()
print(data_normals)

*Risultato:* si ottiene una tabella 12×N (un valore medio per mese e per
stazione), base per il calcolo delle anomalie.

## Calcolo delle anomalie

**Cosa fa:** sottrae a ogni mese la rispettiva normale, ottenendo `data_anom` (le
anomalie di temperatura), e ne traccia la serie per ogni stazione.
**Perché:** le anomalie rimuovono il ciclo stagionale e la climatologia locale,
rendendo le stazioni direttamente confrontabili e sommabili tra loro.

In [ ]:
# calcolo vettorializzato delle anomalie (più efficiente e più veloce)
data_anom = data0.copy()
for month in range(1, 13):
    data_anom[data0.index.month == month] -= data_normals.loc[month]

n = len(metadata0)
fig = plt.figure(figsize=(13, 2.5 * n))

for i, (st_id, row) in enumerate(metadata0.iterrows()):
    # Mappa (sinistra)
    ax_map = plt.subplot(n, 2, i * 2 + 1, projection=ccrs.PlateCarree())
    ax_map.set_title(f"{row.nome} - {row.paese}", fontsize=9)
    ax_map.add_feature(cf.LAND, facecolor="lightgray")
    ax_map.add_feature(cf.OCEAN, facecolor="lightblue")
    ax_map.coastlines(linewidth=0.2)
    ax_map.plot(row.lon, row.lat, "ro", markersize=5, transform=ccrs.Geodetic())

    # Serie temporale anomalie (destra)
    ax_ts = plt.subplot(n, 2, i * 2 + 2)
    if st_id in data_anom.columns:
        ax_ts.plot(data_anom[st_id], color='tab:blue', lw=0.8)
        ax_ts.set_title(f"Anomalie: {st_id}")
        ax_ts.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

The anomaly series show the monthly deviation of temperature from the mean
seasonal cycle. A slightly positive trend is observed at the most recent stations,
consistent with the global warming of the 20th and 21st centuries. Interannual
variability is high, but the long-term signal emerges clearly in the most complete
series.

## Serie regionale unica (Regno Unito + Irlanda)

**Cosa fa / perché:** in origine le stazioni venivano interpolate su una griglia
globale 10°×10° e poi mediate tra le celle. Su una regione piccola come le Isole
Britanniche questo produceva **celle con pochissime stazioni (anche una sola)**,
quindi serie rumorose, con lunghi buchi temporali, che pesavano nella media come
celle ben campionate — rendendo il confronto con l'AMO poco affidabile.

Poiché le Isole Britanniche sono **un'unica regione climatica omogenea**, le
trattiamo come **una sola serie**: si media direttamente su tutte le stazioni.
Le anomalie (in cui la climatologia di ogni stazione è già stata rimossa) sono
confrontabili, quindi la media è lecita e dà una serie **robusta e senza buchi**.
Si mantengono i nomi `data_glob` e `yr_axis` già usati dalle celle successive, in
modo che il resto del notebook continui a funzionare senza altre modifiche.

In [ ]:
fig = plt.figure(figsize=(7, 5))

# Mappa (Top): stazioni UK + Irlanda (al posto delle celle della griglia)
ax = fig.add_subplot(2, 1, 1, projection=ccrs.PlateCarree())
ax.set_extent([-11, 2, 49.5, 61], crs=ccrs.PlateCarree())
ax.add_feature(cf.LAND, facecolor="lightgray")
ax.add_feature(cf.OCEAN, facecolor="lightblue")
ax.coastlines(resolution="10m", linewidth=0.5)
ax.scatter(metadata0.lon, metadata0.lat, color='red', marker='x', s=25,
           transform=ccrs.PlateCarree())

# Serie regionale unica: media diretta su tutte le stazioni
regional_mo = data_anom.mean(axis=1).reindex(taxis)   # media mensile su tutte le stazioni
regional_yr = regional_mo.resample("YE").mean()       # media annuale (1850–2018)

yr_axis   = regional_yr.index      # asse annuale coerente (169 anni)
data_glob = regional_yr.values     # nome riusato da df1['anomUKIE']

# Serie (Bottom)
tser = fig.add_subplot(2, 1, 2)
tser.plot(regional_mo.index, regional_mo, color='gray', lw=0.2, alpha=0.6)  # mensile di sfondo
tser.plot(yr_axis, data_glob, color='red', lw=2)                            # media annuale regionale
tser.axhline(y=0, color='black', linestyle='-')
tser.set_title('Anomalie di temperatura nel Regno Unito e in Irlanda (media regionale in rosso)')
tser.set_ylabel('[°C]')

fig.tight_layout()
plt.show()

*Risultato:* la serie regionale è ora **continua e priva di buchi** (niente più
celle a una sola stazione). La media annuale (rosso) mostra anomalie oscillanti e
prevalentemente sotto lo zero fino a metà ’900, seguite da un **chiaro trend di
riscaldamento dagli anni ’90**, con gli ultimi anni stabilmente sopra lo zero
(~+1 °C).

## Serie annuale delle anomalie regionali

**Cosa fa:** salva la serie annuale regionale in un DataFrame (`df1`, colonna
`anomUKIE`) indicizzato per anno, pronto per il confronto con l'AMO Index.

In [ ]:
df1 = pd.DataFrame(pd.date_range('1850', '2015', freq='YE'), columns = ['anno'])
df1 = pd.DataFrame({'anno': yr_axis.year})
df1['anomUKIE'] = data_glob

# Indice AMO da temperatura del mare (HadISST)

Nella versione precedente l'AMO Index era calcolato dalla **pressione al livello
del mare (PSL)** di un modello: ma quella è, di fatto, una grandezza di
circolazione atmosferica, non l'AMO. L'**Atlantic Multidecadal Oscillation** è
definita sulla **temperatura superficiale del mare (SST)** del Nord Atlantico.

Per questo la parte sulla pressione è stata **rimossa** e sostituita con il calcolo
dell'AMO dalla SST osservata **HadISST** (Met Office Hadley Centre), seguendo la
definizione di Trenberth & Shea: media di area del Nord Atlantico **meno** media
dell'oceano quasi-globale, per rimuovere il segnale di riscaldamento globale e
isolare la componente multidecennale atlantica.

## Caricamento dei dati SST (HadISST)

**Cosa fa:** apre il file NetCDF HadISST e maschera i valori non oceanici.
**Perché:** HadISST usa `-1000` per la terraferma e valori molto negativi per i
dati mancanti: vanno messi a `NaN` per non falsare le medie. La griglia è regolare
1°×1° (lat/lon 1D), quindi semplice da gestire con `xarray`.

> File da scaricare: <https://www.metoffice.gov.uk/hadobs/hadisst/data/HadISST_sst.nc.gz>
> (decomprimere il `.gz` → `HadISST_sst.nc`). HadISST copre il periodo **dal 1870**.

In [ ]:
# File HadISST scaricato e decompresso in locale (HadISST_sst.nc.gz -> HadISST_sst.nc)
sstfile = r'C:\Users\kekko\Downloads\Python\Lab of Geo\HadISST_sst.nc'
ds_sst = xr.open_dataset(sstfile)

# Maschera dei valori non oceanici: -1000 (terra) e valori mancanti molto negativi -> NaN
sst = ds_sst.sst.where(ds_sst.sst > -100)
sst

*Risultato:* `sst` è il campo mensile di temperatura del mare (°C) sull'oceano
globale, con la terraferma mascherata a `NaN`.

## Definizione delle regioni e calcolo delle anomalie

**Cosa fa:** definisce due regioni — il **Nord Atlantico** (lat 0–60°N, lon
−80°..0°, convenzione HadISST −180..180) e l'**oceano quasi-globale** (lat
−60°..60°) — e ne calcola le anomalie mensili rispetto a una climatologia di
riferimento (1870–1900).
**Perché:** l'AMO è il contrasto tra il segnale atlantico e quello globale; le
anomalie rimuovono il ciclo stagionale.

In [ ]:
# Convenzione longitudine HadISST: -180..180
NA = sst.where((ds_sst.latitude > 0) & (ds_sst.latitude < 60) &
               (ds_sst.longitude > -80) & (ds_sst.longitude < 0))   # Nord Atlantico (zona AMO)
GL = sst.where((ds_sst.latitude > -60) & (ds_sst.latitude < 60))    # oceano quasi-globale (riferimento)

# Anomalie mensili rispetto alla climatologia di riferimento 1870-1900
climNA = NA.sel(time=slice('1870-01', '1900-12')).groupby('time.month').mean()
climGL = GL.sel(time=slice('1870-01', '1900-12')).groupby('time.month').mean()
anomNA = NA.groupby('time.month') - climNA
anomGL = GL.groupby('time.month') - climGL

## Indice AMO (media pesata per area)

**Cosa fa:** calcola la media spaziale **pesata per il coseno della latitudine**
(le celle si restringono verso i poli) di ciascuna regione, ne fa la differenza
(Atlantico − globo) e passa alla risoluzione annuale.
**Perché:** la pesatura per area evita di sovrastimare le alte latitudini; la
differenza isola la componente multidecennale atlantica; l'annuale riduce il
rumore. La media pesata ignora i `NaN`, quindi la terraferma è esclusa da sola.

In [ ]:
# Pesi proporzionali all'area delle celle (coseno della latitudine, lat 1D regolare)
w = np.cos(np.deg2rad(ds_sst.latitude))

# AMO Index (SST) = Nord Atlantico - oceano quasi-globale, medie pesate, poi annuale
amo_sst = (anomNA.weighted(w).mean(('longitude', 'latitude'))
           - anomGL.weighted(w).mean(('longitude', 'latitude'))).resample(time='YE').mean()
amo_sst

## Serie annuale dell'AMO Index

**Cosa fa:** salva l'AMO annuale in `df2` (colonna `anomAmo`, indicizzata per
anno), mantenendo lo stesso nome di colonna usato dal confronto finale.

In [ ]:
# DataFrame anno -> valore dell'AMO Index (da SST)
df2 = pd.DataFrame({'anno': pd.DatetimeIndex(amo_sst['time'].values).year})
df2['anomAmo'] = amo_sst.values   # stesso nome 'anomAmo' usato dal confronto a valle
df2 = df2.set_index('anno')
df2.head()

## Grafico dell'AMO Index (da SST)

**Cosa fa:** visualizza l'AMO annuale evidenziando in arancione le fasi positive e
in blu quelle negative.

In [ ]:
plt.figure(figsize=(7, 4))
x = df2['anomAmo']
y = df2.index.values
plt.axhline(0, color='black', linestyle='-')
plt.title('Oscillazione multidecennale atlantica (AMO da SST HadISST)')
plt.xlabel('Anno')
plt.ylabel('AMO Index [°C]')
plt.fill_between(y, x, where=x > 0, color='orange')
plt.fill_between(y, x, where=x < 0, color='blue')
plt.show()

*Risultato atteso* (da verificare rieseguendo in locale con HadISST): l'indice
mostra le tipiche **fasi positive e negative su scale di 20–40 anni** dell'AMO,
coerenti con la letteratura (fasi calde indicative anni ’30–’60 e dagli anni ’90,
fase fredda anni ’70–’80).

## Unione delle due serie

**Cosa fa:** unisce in un unico DataFrame `finale` la serie annuale delle anomalie
UK+Irlanda (`anomUKIE`) e l'AMO Index da SST (`anomAmo`), allineandole per anno.

In [ ]:
# Unione delle anomalie regionali (anomUKIE) con l'AMO Index (anomAmo), per anno
df2.reset_index(inplace = True)
finale = df1.merge(df2, how='left')
finale = finale.set_index('anno')


## Confronto anomalie (Regno Unito + Irlanda) vs AMO Index: scelta del taglio temporale

**Cosa fa / perché:** il confronto richiede una **finestra temporale comune e ben
campionata**:

* la SST HadISST parte dal **1870**;
* prima del **1900** le stazioni delle Isole Britanniche sono pochissime, quindi la
  media regionale è rumorosa e poco rappresentativa;
* entrambe le serie sono affidabili fino al **2018** (ultimo anno delle anomalie
  UK+Irlanda).

Il confronto viene quindi **ritagliato al periodo 1900–2018** e si usa una media
mobile **centrata** a 15 anni per isolare il segnale multidecennale (l'AMO ha
periodi di 20–40 anni) ed eliminare la variabilità interannuale, rendendo il
confronto leggibile e logicamente coerente.

In [ ]:
# Confronto tra le anomalie di temperatura (Regno Unito + Irlanda) e l'AMO Index (da SST).
# Finestra comune e ben campionata: HadISST parte dal 1870; prima del 1900 le
# stazioni UK+IRL sono troppo poche; le anomalie UK+IRL arrivano al 2018.
START, END = 1900, 2018
conf = finale.loc[START:END]

plt.figure(figsize=(10, 4))

# media mobile CENTRATA a 15 anni: isola il segnale multidecennale (l'AMO ha
# periodi tipici di 20-40 anni) eliminando la variabilita' interannuale.
ax3 = conf.anomAmo.rolling(window=15, center=True).mean().plot(
    color='green', grid=True, label='AMO Index (SST)')
ax4 = conf.anomUKIE.rolling(window=15, center=True).mean().plot(
    color='red', grid=True, secondary_y=True, label='Anomalie UK+IRL')

plt.title('Confronto tra anomalie di temperatura (Regno Unito + Irlanda) e AMO Index da SST (1900-2018)')
ax3.set_xlabel('Anno')
ax3.set_ylabel('AMO Index [°C]', color='green')
ax4.set_ylabel('Anomalie UK+IRL [°C]', color='red')

h1, l1 = ax3.get_legend_handles_labels()
h2, l2 = ax4.get_legend_handles_labels()
ax3.legend(h1 + h2, l1 + l2, loc='upper left')

plt.show()

## Osservazioni sul confronto

Confrontando le due serie nel periodo 1900–2018, le anomalie di temperatura delle
Isole Britanniche (Regno Unito + Irlanda) tendono a seguire le **fasi multidecennali
dell'AMO**: alle fasi positive dell'AMO corrispondono in genere periodi più caldi
nell'area, a quelle negative periodi più freddi. Avendo ora calcolato l'AMO dalla
**SST** (definizione fisica corretta) anziché dalla pressione, la corrispondenza tra
i due segnali è più interpretabile. Sovrapposto alla modulazione dell'AMO resta
visibile un **trend di fondo di riscaldamento** nella seconda metà del XX secolo,
coerente con il riscaldamento globale di origine antropica.

*(I valori numerici esatti vanno verificati rieseguendo il notebook in locale con i
dati CRUTEM e HadISST.)*